In [1]:
import os
import networkx as nx
from rdkit import Chem
from karateclub.estimator import Estimator
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from karateclub.utils.treefeatures import WeisfeilerLehmanHashing
import numpy as np
from collections import Counter
from ksvd import ApproximateKSVD

C:\Users\ASUS\OneDrive\Desktop\UoR\FoE\FYP\project\Research\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class WL_KSVD(Estimator):
    r""" An implementation of WL_KSVD

    Args:
        wl_iterations (int): Number of Weisfeiler-Lehman iterations. Default is 2.
        attributed (bool): Presence of graph attributes. Default is False.
        dimensions (int): Dimensionality of embedding. Default is 128.
        workers (int): Number of cores. Default is 4.
        down_sampling (float): Down sampling frequency. Default is 0.0001.
        epochs (int): Number of epochs. Default is 10.
        learning_rate (float): HogWild! learning rate. Default is 0.025.
        min_count (int): Minimal count of graph feature occurrences. Default is 5.
        seed (int): Random seed for the model. Default is 42.
        erase_base_features (bool): Erasing the base features. Default is False.

        n_vocab: Number of preliminary vocabulary size.  Default is 1000
        n_atoms: Number of dictionary elements (atoms). Default is 128
        n_non_zero_coefs: Number of nonzero coefficients to target. Default is 10
        max_iter: Maximum number of iterations. Default is 10
        tol: Tolerance for error. Default is 1e-6

    """

    def __init__(
        self,
        wl_iterations: int = 2,
        attributed: bool = False,
        dimensions: int = 128,
        workers: int = 4,
        down_sampling: float = 0.0001,
        epochs: int = 10,
        learning_rate: float = 0.025,
        min_count: int = 5,
        seed: int = 42,
        erase_base_features: bool = False,
        n_vocab: int = 1000,
        n_atoms: int = 128,
        n_non_zero_coefs: int = 10,
        max_iter: int = 10,
        tol: float = 1e-6

    ):
        self.wl_iterations = wl_iterations
        self.attributed = attributed
        self.dimensions = dimensions
        self.workers = workers
        self.down_sampling = down_sampling
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.min_count = min_count
        self.seed = seed
        self.erase_base_features = erase_base_features
        self.n_vocab = n_vocab
        self.n_atoms = n_atoms
        self.n_non_zero_coefs = n_non_zero_coefs
        self.max_iter = max_iter
        self.tol = tol

In [4]:
print('Loading NCI1 dataset')

graphs = []
y = []

filepath = "datasets/NCI_full/1total-connect.sdf"

supplier = Chem.SDMolSupplier(filepath, sanitize=False, removeHs=False)
for mol in supplier:
    if mol is None:
        continue

    G = nx.Graph()

    # Add atoms as nodes
    for atom in mol.GetAtoms():
        G.add_node(
            atom.GetIdx(),
            feature=atom.GetSymbol()   # WL uses node labels
        )

    # Add bonds as edges
    for bond in mol.GetBonds():
        G.add_edge(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx()
        )

    # Get graph label
    # In NCI1, class label is stored as a molecule property
    label = int(float(mol.GetProp("value")))
    graphs.append(G)


    y.append(label)

print(f"Loaded {len(graphs)} graphs")

Loading NCI1 dataset


[21:48:39] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[21:48:41] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[21:48:45] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[21:48:47] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[21:48:51] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[21:48:51] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.


Loaded 37349 graphs


In [6]:
wl_ksvd = WL_KSVD()

In [7]:
import time
start = time.perf_counter()
documents = []
# TODO: parallel implementation
for graph in graphs:
    g = wl_ksvd._check_graph(graph)
    document = WeisfeilerLehmanHashing(
        g, wl_ksvd.wl_iterations, wl_ksvd.attributed, wl_ksvd.erase_base_features)

    documents.append(document)
end = time.perf_counter()
t = end - start

In [8]:
t

21.88612489999997

Parallelize the Hash Generation

In [13]:
from joblib import Parallel, delayed

start = time.perf_counter()

def process_graph(graph):
    g = wl_ksvd._check_graph(graph)
    document = WeisfeilerLehmanHashing(
        g, wl_ksvd.wl_iterations, wl_ksvd.attributed, wl_ksvd.erase_base_features)

    return document

raw_documents = Parallel(n_jobs=wl_ksvd.workers)(delayed(process_graph)(g) for g in graphs)

end = time.perf_counter()
t = end - start

In [14]:
t

17.1934215

In [15]:
len(documents)

37349

In [16]:
len(raw_documents)

37349

In [19]:
documents[0].get_graph_features()

['3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '4025f7ec584c225eb214f61e8fa3aaba',
 '4',
 '9ded9482dcd25745e629e5fd3afbca73',
 '7f01cbf69516028cdff263ab4ef5c7ee',
 '4',
 'af67ad2ffa467c8bad3f840d7c715bb0',
 'dfd14c0b867116b2789a3cc87801ee0d',
 '4',
 'af67ad2ffa467c8bad3f840d7c715bb0',
 '4154eddd1c77b6f2435115470383c073',
 '3',
 '83330c87ecc2517466a49ffe3555fd98',
 '26748bda89269e5edc7bcfea13c999ac',
 '4',
 '8f9e9234712a641a944b115e2f8d07f3',
 '5b05083c2746f603211da50b62c0c45a',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '796097a7389b46e8234f8bc85fa962fa',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '0c823a59055acdfdb3984bf1e8af3203',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '0c823a59055acdfdb3984bf1e8af3203',
 '4',
 '8f9e9234712a641a944b115e2f8d07f3',
 'b1fb9cffdbc26abd1ebceb04ce119249',
 '4',
 '9ded9482dcd25745e629e5fd3afbca73',
 'c56ff23927b49a2444f3b2f6608992a6',
 '5',
 '5f72132389c5a8352ab504ead192f1ac',
 'f96a606c096ff671e73ae6288eb3f91c',
 '6',
 '140a8e01d81b435e04faf07c3c650dc3

In [20]:
raw_documents[0].get_graph_features()

['3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '4025f7ec584c225eb214f61e8fa3aaba',
 '4',
 '9ded9482dcd25745e629e5fd3afbca73',
 '7f01cbf69516028cdff263ab4ef5c7ee',
 '4',
 'af67ad2ffa467c8bad3f840d7c715bb0',
 'dfd14c0b867116b2789a3cc87801ee0d',
 '4',
 'af67ad2ffa467c8bad3f840d7c715bb0',
 '4154eddd1c77b6f2435115470383c073',
 '3',
 '83330c87ecc2517466a49ffe3555fd98',
 '26748bda89269e5edc7bcfea13c999ac',
 '4',
 '8f9e9234712a641a944b115e2f8d07f3',
 '5b05083c2746f603211da50b62c0c45a',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '796097a7389b46e8234f8bc85fa962fa',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '0c823a59055acdfdb3984bf1e8af3203',
 '3',
 '9dafdb095db58e4fb4a689fb082bb849',
 '0c823a59055acdfdb3984bf1e8af3203',
 '4',
 '8f9e9234712a641a944b115e2f8d07f3',
 'b1fb9cffdbc26abd1ebceb04ce119249',
 '4',
 '9ded9482dcd25745e629e5fd3afbca73',
 'c56ff23927b49a2444f3b2f6608992a6',
 '5',
 '5f72132389c5a8352ab504ead192f1ac',
 'f96a606c096ff671e73ae6288eb3f91c',
 '6',
 '140a8e01d81b435e04faf07c3c650dc3